## Estimating Equilibrium Climate Sensitivity (ECS) in CMIP6 Models

**Adapted from** https://github.com/pangeo-gallery/cmip6/blob/master/ECS_Gregory_method.ipynb

In this exercise, you will be estimating the Equilibrium Climate Sensitivity (ECS) of the CMIP6 models using the ["Gregory Method"](https://agupubs.onlinelibrary.wiley.com/doi/epdf/10.1029/2003GL018747). ECS is defined as the change in global mean near surface air temperature (tas) due to a doubling of CO$_{2}$ concentrations, once the climate system has reached equilibrium (i.e. the net radiative imbalance at the top-of-atmosphere (TOA) is zero). You will be working with global, annual mean tas and TOA radiation data from the `abrupt-4xCO2` and `piControl` experiments in the CMIP6 archive. We would typically use the `abrupt-2xCO2` data for this method, but we'll be using `abrupt-4xCO2` because there is more data available. The global mean data will be provided to you in a netCDF format, to save time. In the end, you will be creating a 4-panel plot, comparing the timeseries of global mean temperature and radiative imbalance across the models, as well as the spread in ECS across the CMIP6 ensemble.

### Required Files

+ abrupt-4xCO2_global_annual_means.nc

### Load CMIP6 global mean data

The first step will be to grab the global mean data for the `abrupt-4xCO2` and `piControl` experiments (which has already been downloaded and pre-processed, using the methods from the previous module). The dataset contains the following variables for each model (`source_id`) and experiment (`experiment_id`):

+ `tas` - near surface air temperature (K)
+ `rsdt` - incoming shortwave radiation at the TOA (W m$^{-2}$)
+ `rsut` - outgoing (reflected) shortwave radiation at the TOA (W m$^{-2}$)
+ `rlut` - outgoing longwave radiation at the TOA (W m$^{-2}$)

In [ ]:
### Load the necessary packages ###
from matplotlib import pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd

### Load CMIP6 data from nc file ###
CMIP6_fil = 'abrupt-4xCO2_global_annual_means.nc'
big_ds = xr.open_dataset(CMIP6_fil)

print(big_ds)

### Calculate Derived Variables

Before we can estimate ECS, we need to calculate two derived quantities:

1) The net radiative imbalance at the TOA, which is the incoming shortwave radiation minus the outgoing shortwave and longwave radiation (imbalance = rsdt - rsut - rlut). A positive imbalance means the Earth is gaining energy.
2) The anomaly of the `abrupt-4xCO2` run compared to the `piControl` run. We calculate the long-term mean of the `piControl` run for each model, and subtract it from both experiments. This removes each model's own baseline climate, so that we are only looking at the response to the CO$_{2}$ forcing.

In [ ]:
### Calculate the net radiative imbalance at the TOA ###
big_ds['imbalance'] = big_ds['rsdt'] - big_ds['rsut'] - big_ds['rlut'] # incoming shortwave - outgoing shortwave - outgoing longwave

### Calculate anomalies relative to the piControl run ###
ds_mean = big_ds[['tas', 'imbalance']].sel(experiment_id='piControl').mean(dim='year') # long-term piControl mean for each model
ds_anom = big_ds[['tas', 'imbalance']] - ds_mean # subtract the piControl mean from both experiments

### Add some metadata ###
ds_anom.tas.attrs['long_name'] = 'Global Mean Surface Temp Anom'
ds_anom.tas.attrs['units'] = 'K'
ds_anom.imbalance.attrs['long_name'] = 'Global Mean Radiative Imbalance'
ds_anom.imbalance.attrs['units'] = 'W m$^{-2}$'

print(ds_anom)

### Select the first 150 years, and store in a Pandas DataFrame

The models cover different time intervals, so we are going to limit our analysis to the first 150 years after the abrupt quadrupling of CO$_{2}$ (the standard period used for the Gregory Method). We will store the `abrupt-4xCO2` anomalies for each model in a Pandas DataFrame. Note that the units of 'tas' are now an anomaly in degrees Kelvin, and 'imbalance' is in W m$^{-2}$:

In [ ]:
### Select the first 150 years of the abrupt-4xCO2 experiment ###
ds_abrupt = ds_anom.sel(year=slice(0,149), experiment_id='abrupt-4xCO2') # select the Gregory 150-year period

### Convert abrupt-4xCO2 dataset into Pandas Dataframe ###
df_abrupt = ds_abrupt.to_dataframe().reset_index()
df_abrupt = df_abrupt[['source_id','year','tas','imbalance']] # we only want the model, year and the timeseries for tas and imbalance
df_abrupt = df_abrupt.rename(columns={'source_id':'model'}) #rename 'source_id' column to 'model'
df_abrupt = df_abrupt.dropna() # some models don't have all 150 years, so drop those missing values

print(df_abrupt)

### Calculate ECS

Right after CO$_{2}$ is quadrupled, there is a large positive radiative imbalance at the TOA. As the planet warms, it radiates more energy back to space, so the imbalance shrinks over time. If we plot the imbalance ($N$) against the temperature anomaly ($\Delta T$), we get a roughly straight line:

$$N = F + \lambda \Delta T$$

where the intercept $F$ is the radiative forcing from 4xCO$_{2}$, and the slope $\lambda$ is the climate feedback parameter (which is negative). The line crosses $N = 0$ (equilibrium) at $\Delta T = -F/\lambda$, which is the equilibrium warming for a quadrupling of CO$_{2}$. Since the radiative forcing of CO$_{2}$ is roughly a logarithmic function of its concentration, a quadrupling is two doublings, so we divide this by 2 to get the ECS. We will fit this line for each model using `np.polyfit`:

In [ ]:
### Create function to calculate ECS for each model ###
def calc_ecs(df): # function which fits a line to imbalance vs. tas and calculates ECS
    slope, intercept = np.polyfit(df['tas'], df['imbalance'], 1) # slope is the feedback parameter (lambda), intercept is the forcing (F)
    return -0.5 * intercept / slope # warming where the imbalance is zero, divided by 2 to go from 4xCO2 to 2xCO2

### Calculate ECS for each model ###
df_ecs = df_abrupt.groupby('model')[['tas','imbalance']].apply(calc_ecs).reset_index(name='ECS') # apply calc_ecs to each model
df_ecs = df_ecs.sort_values('ECS') # sort models from lowest to highest ECS

print(df_ecs)

### Plot the Gregory Regression for each Model

Let's take a look at the relationship between the radiative imbalance and the temperature anomaly for each model, along with the fitted line, extended out to where it crosses zero. This is similar to a plot made by [Mark Zelinka](https://twitter.com/mzelinka/status/1255534531144085513):

In [ ]:
### Load Plotting Tools (Seaborn) ###
import seaborn as sns

### Create scatter plot and line of best fit for each model ###
g = sns.lmplot(data=df_abrupt, x='tas', y='imbalance', col='model', col_wrap=6, height=2.5,
               ci=None, truncate=False, # don't plot a confidence interval, and extend the line of best fit to the edges of the plot
               scatter_kws={'s':5}, line_kws={'color':'black'},
               facet_kws={'xlim':(0,10), 'ylim':(-2,10)})

### Add the ECS value to each panel ###
ecs_by_model = df_ecs.set_index('model')['ECS'] # look up ECS by model name
for model, ax in g.axes_dict.items():
    ax.axhline(0, color='grey', linestyle='--') # radiative balance (equilibrium)
    ax.text(3, 8, f'ECS = {ecs_by_model[model]:3.2f} K', fontweight='bold', fontsize=10)
    ax.grid()
g.set_axis_labels('Temp Anom (K)', 'Imbalance (W m$^{-2}$)')

plt.show()

### Create Summary Figures

At this point it is time to investigate the ensemble mean and spread in the CMIP6 response to an abrupt quadrupling of CO$_{2}$. You will be creating a 4-panel figure. The top 2 panels will show the timeseries of the global mean temperature anomaly and radiative imbalance (ensemble mean and 95% confidence interval), while the bottom 2 panels will show the distribution of ECS across the CMIP6 models as a boxplot, and the ECS of each individual model:

In [ ]:
### Create Figures ###
fig,axs = plt.subplots(nrows=2,ncols=2,figsize=(15,15)) #initialize a 4 panel figure

mean_style={"marker":"D", "markerfacecolor":"black", "markeredgecolor":"black", "markersize":"5"} #customize the display of the ensemble mean in the boxplot
flierprops = dict(marker='d', markersize=5) #customize the display of the outliers in the boxplot

## tas timeseries ##
ax1 = plt.subplot(221)
sns.lineplot(data=df_abrupt, x="year", y="tas", errorbar=('ci', 95), ax=ax1) #use the 95% confidence interval
ax1.set_ylabel('Near Surface Air Temp Anom (Kelvin)',fontsize=14,fontweight='bold')
ax1.set_xlabel('Year',fontsize=14,fontweight='bold')

## imbalance timeseries ##
ax2 = plt.subplot(222)
sns.lineplot(data=df_abrupt, x="year", y="imbalance", errorbar=('ci', 95), ax=ax2) #use the 95% confidence interval
ax2.set_ylabel('Radiative Imbalance (W m$^{-2}$)',fontsize=14,fontweight='bold')
ax2.set_xlabel('Year',fontsize=14,fontweight='bold')

### ECS boxplot ###
ax3 = plt.subplot(223)
sns.boxplot(data=df_ecs, y="ECS", showmeans=True, meanprops=mean_style, flierprops=flierprops, ax=ax3)
ax3.set_ylabel('ECS (Kelvin)',fontsize=14,fontweight='bold')
ax3.set_xlabel('') #no x-label needed for boxplots
ax3.set_xticks([]) #remove x-ticks

### ECS for each model ###
ax4 = plt.subplot(224)
sns.barplot(data=df_ecs, x="model", y="ECS", color='tab:blue', ax=ax4) #models are sorted from lowest to highest ECS
ax4.set_ylabel('ECS (Kelvin)',fontsize=14,fontweight='bold')
ax4.set_xlabel('') #model names are shown on the x-ticks
ax4.tick_params(axis='x', labelrotation=90) #rotate model names so they don't overlap

plt.show()

### Reflection Questions

In this module, you estimated the Equilibrium Climate Sensitivity of the CMIP6 models using the Gregory Method, applied to the `abrupt-4xCO2` experiment. We leave you with a couple of reflection questions:

1) What is the range of ECS across the CMIP6 models, and how does the ensemble mean compare to the IPCC AR6 assessed likely range of 2.5 to 4 K? What could cause some models to have a much higher ECS than others? (hint: think about the climate feedbacks discussed in the text)
2) Look back at the Gregory regression plots for the individual models. Is the relationship between the radiative imbalance and the temperature anomaly perfectly linear over the 150 years? How might this affect the ECS estimated by the Gregory Method?